# 02_Linh – QDRANT

## Mục tiêu
- Đưa dữ liệu sản phẩm TV từ `merged_data.csv` vào Qdrant.
- Tạo vector embedding từ thông tin sản phẩm.
- Kiểm tra collection và số lượng dữ liệu đã lưu.
- Thực hiện tìm kiếm tương đồng (vector search) trên Qdrant và hiển thị kết quả.

**Dữ liệu đầu vào:** `merged_data.csv`  
**Database:** Qdrant  
**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("merged_data.csv")

print("Kích thước dữ liệu:", df.shape)
display(df.head())


## 1. Chuẩn bị dữ liệu để tạo embedding

Sử dụng các thông tin mô tả sản phẩm như tên sản phẩm, loại TV, kích thước màn hình, độ phân giải, hệ điều hành và hãng sản xuất để tạo chuỗi văn bản đại diện cho mỗi sản phẩm.


In [ ]:
text_columns = [
    "product_name", "tv_type", "screen_size", "resolution",
    "screen_type", "operating_system", "manufacturer",
    "model_name"
]

existing_columns = [c for c in text_columns if c in df.columns]

df["text"] = df[existing_columns].fillna("").astype(str).agg(" | ".join, axis=1)

display(df[["product_name", "text"]].head())


## 2. Tạo embedding bằng SentenceTransformer


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df["text"].tolist(),
    show_progress_bar=True
)

embeddings = np.asarray(embeddings)

print("Số vector:", len(embeddings))
print("Kích thước mỗi vector:", embeddings.shape[1])


## 3. Kết nối và tạo collection trong Qdrant


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(
    url="http://localhost:6333"
)

collection_name = "tv_products"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=embeddings.shape[1],
        distance=Distance.COSINE
    )
)

print("Đã tạo collection:", collection_name)


## 4. Đưa dữ liệu vào Qdrant bằng Upsert

Mỗi sản phẩm được lưu cùng:
- vector embedding;
- tên sản phẩm;
- giá;
- các thuộc tính sản phẩm;
- nội dung `text`.


In [ ]:
from qdrant_client.models import PointStruct

points = []

for i, row in df.iterrows():
    payload = {
        "product_name": row.get("product_name"),
        "price": row.get("price"),
        "tv_type": row.get("tv_type"),
        "screen_size": row.get("screen_size"),
        "resolution": row.get("resolution"),
        "operating_system": row.get("operating_system"),
        "manufacturer": row.get("manufacturer"),
        "model_name": row.get("model_name"),
        "text": row.get("text")
    }

    points.append(
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload=payload
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)

print("Đã upsert:", len(points), "records")


## 5. Kiểm tra collection sau khi ingest


In [ ]:
info = client.get_collection(collection_name)

print("Collection:", collection_name)
print("Số lượng vector đã lưu:", info.points_count)
print("Kích thước vector:", info.config.params.vectors.size)


## 6. Query / Vector Search

Thử tìm các sản phẩm có nội dung tương đồng với một câu truy vấn. Đây là bước kiểm tra khả năng tìm kiếm dữ liệu sau khi ingest vào Qdrant.


In [ ]:
query = "Smart TV 55 inch 4K"

query_vector = model.encode(query).tolist()

search_result = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5,
    with_payload=True
).points

results = []

for hit in search_result:
    results.append({
        "score": hit.score,
        "product_name": hit.payload.get("product_name"),
        "price": hit.payload.get("price"),
        "tv_type": hit.payload.get("tv_type"),
        "screen_size": hit.payload.get("screen_size"),
        "resolution": hit.payload.get("resolution"),
        "manufacturer": hit.payload.get("manufacturer")
    })

result_df = pd.DataFrame(results)
display(result_df)


### So What?

Kết quả tìm kiếm cho thấy Qdrant có thể truy vấn các sản phẩm dựa trên mức độ tương đồng của vector embedding. Cách lưu trữ này phù hợp cho các bài toán tìm kiếm ngữ nghĩa hoặc xây dựng hệ thống AI/RAG trên dữ liệu sản phẩm.


## 7. Kết luận

- Dữ liệu từ `merged_data.csv` đã được chuyển thành vector embedding.
- Các vector và thông tin sản phẩm đã được lưu vào collection `tv_products` của Qdrant.
- Đã kiểm tra số lượng vector sau khi ingest.
- Đã thực hiện vector search và lấy ra các sản phẩm tương đồng với câu truy vấn.
